# Phase 4: Fusion-level SHAP Analysis

Quantifies **modality contribution** (text-origin vs image-origin) to each target score
using SHAP (SHapley Additive exPlanations) on the fused embedding.

| Component | Configuration |
|---|---|
| Fusion | CrossAttentionFusion — fused [B, 1024] (512 text-origin + 512 image-origin) |
| Head | Linear(1024→512)→ReLU→DO→Linear(512→256)→ReLU→Linear(256→5) |
| SHAP Method | DeepExplainer (gradient-based, fast) |
| Background | 100 samples from validation set |

**Central question:** Was the model more influenced by image or text for each target?

---
### STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### STEP 2: Clone and install

In [ ]:
!rm -rf /content/SE365
!git clone -b xai-v3 https://github.com/lechihoang/SE365.git /content/SE365
%cd /content/SE365
!pip install -q -r requirements.txt
!pip install -q shap

### STEP 3: Download and extract data

In [ ]:
!rm -rf ./data
!cp /content/drive/MyDrive/SE365/data.zip ./data.zip
!unzip -q data.zip
!rm data.zip
!ls -la ./data

### STEP 4: Configuration

In [ ]:
import os, sys, time, warnings
warnings.filterwarnings('ignore')

DRIVE_ROOT   = '/content/drive/MyDrive/SE365'
PROJECT_ROOT = '/content/SE365'
EXP_ID       = 'EXP_060A_bestsequential_full_configuration'

EXP_DIR      = f'{DRIVE_ROOT}/experiments/{EXP_ID}'
XAI_OUT_DIR  = f'{EXP_DIR}/xai/shap'
DATA_DIR     = f'{PROJECT_ROOT}/data/text'
IMAGE_DIR    = f'{PROJECT_ROOT}/data/image'

NUM_SHAP_SAMPLES = 15
N_BACKGROUND     = 100

os.makedirs(XAI_OUT_DIR, exist_ok=True)
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'PROJECT_ROOT    : {PROJECT_ROOT}')
print(f'EXP_DIR         : {EXP_DIR}')
print(f'XAI_OUT_DIR     : {XAI_OUT_DIR}')
print(f'NUM_SHAP_SAMPLES: {NUM_SHAP_SAMPLES}')
print(f'N_BACKGROUND    : {N_BACKGROUND}')

### STEP 5: Imports and Seed

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 4 — Step 5 — Imports and Seed')
print('='*60)

import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from xai.config import (
    TARGET_NAMES, FACTOR_NAMES, DISPLAY_NAMES, NUM_TARGETS,
    FUSED_DIM, CROSS_ATTN_HIDDEN_DIM,
    DEFAULT_SEED, DEFAULT_DPI,
    BEST_TEXT_MODEL, BEST_IMAGE_MODEL,
    COLOR_SCHEMES,
)
from xai.utils import (
    get_device, set_seed, get_tokenizer, get_image_processor,
    load_model, load_single_sample, get_prediction,
    save_raw_values, get_metadata,
)
from xai.shap_explainer import (
    SHAPExplainer, FusionHeadWrapper,
    extract_fused_embeddings, select_background,
    compute_shap_values, modality_contribution,
    additivity_check, run_ablation_check,
    plot_modality_contribution, plot_modality_single_target,
)

SEED = DEFAULT_SEED
set_seed(SEED)
device = get_device()

TEXT_DIM = CROSS_ATTN_HIDDEN_DIM  # 512 — text-origin dims in fused vector

print(f'Device   : {device}')
print(f'Seed     : {SEED}')
print(f'TEXT_DIM : {TEXT_DIM} (fused dims 0:{TEXT_DIM} = text-origin)')
print(f'Done ({time.time()-t0:.1f}s)')

### STEP 6: Load Model

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 4 — Step 6 — Load Model')
print('='*60)

model, config = load_model(EXP_DIR, device=device)

# Verify head structure
print(f'Model : {model.__class__.__name__}')
print(f'Head  : {model.head}')
print(f'Done ({time.time()-t0:.1f}s)')

### STEP 7: Create DataLoader for Embedding Extraction

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 4 — Step 7 — Create DataLoader')
print('='*60)

text_model_name = config.get('text_model_name', BEST_TEXT_MODEL)
image_model_name = config.get('image_model_name', BEST_IMAGE_MODEL)
tokenizer = get_tokenizer(text_model_name)
image_processor = get_image_processor(image_model_name)

from src.dataset import MultimodalDataset

# Use validation set for background, test set for explanation
val_csv = os.path.join(DATA_DIR, 'val.csv')
test_csv = os.path.join(DATA_DIR, 'test.csv')
SPLIT_CSV = test_csv if os.path.isfile(test_csv) else val_csv
SPLIT_NAME = 'test' if SPLIT_CSV == test_csv else 'validation'

val_dataset = MultimodalDataset(
    val_csv, tokenizer, image_processor,
    max_length=config.get('max_length', 256),
    image_dir=IMAGE_DIR,
)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)

test_dataset = MultimodalDataset(
    SPLIT_CSV, tokenizer, image_processor,
    max_length=config.get('max_length', 256),
    image_dir=IMAGE_DIR,
)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0)

print(f'Val samples  : {len(val_dataset)}')
print(f'Test samples : {len(test_dataset)}')
print(f'Split        : {SPLIT_NAME}')
print(f'Done ({time.time()-t0:.1f}s)')

### STEP 8: Extract Fused Embeddings

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 4 — Step 8 — Extract Fused Embeddings')
print('='*60)

# Extract from validation (for background)
print('Extracting validation embeddings...')
val_fused, val_labels, val_preds = extract_fused_embeddings(
    model, val_loader, device, max_samples=min(500, len(val_dataset))
)
print(f'Val fused : {val_fused.shape}')

# Extract from test (for explanation)
print('Extracting test embeddings...')
test_fused, test_labels, test_preds = extract_fused_embeddings(
    model, test_loader, device, max_samples=min(NUM_SHAP_SAMPLES + 50, len(test_dataset))
)
print(f'Test fused: {test_fused.shape}')

# Verify prediction reproduction
with torch.no_grad():
    reproduced = model.head(val_fused[:5].to(device))
repro_err = (reproduced.cpu() - val_preds[:5]).abs().max().item()
print(f'Prediction reproduction error: {repro_err:.2e}')
assert repro_err < 1e-4, f'Reproduction error too large: {repro_err}'
print(f'Prediction reproduction: PASSED')

# Save embeddings
raw_dir = os.path.join(XAI_OUT_DIR, 'raw')
os.makedirs(raw_dir, exist_ok=True)
torch.save(val_fused, os.path.join(raw_dir, 'val_fused_embeddings.pt'))
torch.save(test_fused, os.path.join(raw_dir, 'test_fused_embeddings.pt'))
print(f'Saved: {raw_dir}/val_fused_embeddings.pt')
print(f'Saved: {raw_dir}/test_fused_embeddings.pt')

print(f'Done ({time.time()-t0:.1f}s)')

### STEP 9: Select Background Samples

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 4 — Step 9 — Select Background')
print('='*60)

background, bg_indices = select_background(
    val_fused, n_background=N_BACKGROUND, seed=SEED
)
print(f'Background : {background.shape}')
print(f'Mean       : {background.mean():.4f}')
print(f'Std        : {background.std():.4f}')

# Save
torch.save(background, os.path.join(raw_dir, 'background_fused.pt'))
save_raw_values({'indices': bg_indices}, os.path.join(raw_dir, 'background_indices.json'))

print(f'Done ({time.time()-t0:.1f}s)')

### STEP 10: Single-Sample SHAP Demo

Compute SHAP values for one sample across all 5 targets.

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 4 — Step 10 — Single-Sample SHAP Demo')
print('='*60)

demo_fused = test_fused[0:1]  # [1, 1024]
demo_pred = test_preds[0]      # [5]
demo_gt = test_labels[0]       # [5]

print(f'Predictions: {[f"{v:.3f}" for v in demo_pred.tolist()]}')
print(f'Ground truth: {[f"{v:.2f}" for v in demo_gt.tolist()]}')

# Compute SHAP for each target
demo_contributions = {}
for t_idx in range(NUM_TARGETS):
    wrapper = FusionHeadWrapper(model.head, score_index=t_idx)
    shap_vals, base_val = compute_shap_values(
        wrapper, background.to(device), demo_fused.to(device)
    )
    contrib = modality_contribution(shap_vals[0], text_dim=TEXT_DIM)
    demo_contributions[FACTOR_NAMES[t_idx]] = contrib

    # Additivity check
    passes, err = additivity_check(wrapper, demo_fused.to(device), shap_vals[0], base_val)

    print(f'  {FACTOR_NAMES[t_idx]:>10s}: text-origin={contrib["text_pct"]:.1f}%  '
          f'image-origin={contrib["image_pct"]:.1f}%  '
          f'additivity={"OK" if passes else "FAIL"} (err={err:.4f})')

print(f'\nDone ({time.time()-t0:.1f}s)')

### STEP 11: Modality Contribution Visualization

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 4 — Step 11 — Modality Contribution Chart')
print('='*60)

fig = plot_modality_contribution(
    contributions=demo_contributions,
    target_names=FACTOR_NAMES,
)
plt.show()

print(f'Done ({time.time()-t0:.1f}s)')

### STEP 12: Ablation Check

Zero out text-origin or image-origin features and measure prediction change.

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 4 — Step 12 — Ablation Check')
print('='*60)

bg_mean = background.mean(dim=0).to(device)
ablation = run_ablation_check(
    model_head=model.head,
    sample_fused=demo_fused[0].to(device),
    background_mean=bg_mean,
    text_dim=TEXT_DIM,
)

print(f'\n{"Target":<15s} {"Full":>8s} {"No Text":>8s} {"No Image":>8s} {"Text Drop%":>10s} {"Image Drop%":>11s}')
print('-'*65)
for t_idx, name in enumerate(FACTOR_NAMES):
    full = ablation['full_prediction'][t_idx]
    no_t = ablation['text_zeroed_prediction'][t_idx]
    no_i = ablation['image_zeroed_prediction'][t_idx]
    t_drop = ablation['text_drop_pct'][t_idx]
    i_drop = ablation['image_drop_pct'][t_idx]
    print(f'{name:<15s} {full:8.3f} {no_t:8.3f} {no_i:8.3f} {t_drop:9.1f}% {i_drop:10.1f}%')

print(f'\nDone ({time.time()-t0:.1f}s)')

### STEP 13: Full Explanation with SHAPExplainer

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 4 — Step 13 — Full Sample Explanation')
print('='*60)

explainer = SHAPExplainer(model=model, device=device, output_dir=XAI_OUT_DIR)

result = explainer.explain_sample(
    sample_fused=demo_fused[0],
    sample_id='sample_0000',
    background=background,
)

print(f'\nArtifacts:')
for name, path in result.get('paths', {}).items():
    print(f'  {name}: {path}')

print(f'\nDone ({time.time()-t0:.1f}s)')

### STEP 14: Batch Processing — 15 Samples

In [ ]:
t0 = time.time()
print('='*60)
print(f'  Phase 4 — Step 14 — Batch Processing ({NUM_SHAP_SAMPLES} samples)')
print('='*60)

# Select samples
n_avail = min(NUM_SHAP_SAMPLES, test_fused.shape[0])
sample_ids = [f'sample_{i:04d}' for i in range(n_avail)]
samples_fused = test_fused[:n_avail]

batch_result = explainer.explain_batch(
    samples_fused=samples_fused,
    sample_ids=sample_ids,
    background=background,
)

# Show aggregate modality contribution
# aggregate is a list of 5 dicts (one per target, indexed by position)
agg_list = batch_result.get('aggregate', [])
if agg_list:
    print(f'\n--- Aggregate Modality Contribution ---')
    print(f'{"Target":<15s} {"Text-Origin%":>13s} {"Image-Origin%":>14s}')
    print('-'*44)
    for t_idx, name in enumerate(FACTOR_NAMES):
        if t_idx < len(agg_list):
            t_pct = agg_list[t_idx].get('text_pct', 0)
            i_pct = agg_list[t_idx].get('image_pct', 0)
            print(f'{name:<15s} {t_pct:12.1f}% {i_pct:13.1f}%')

print(f'\nBatch complete: {n_avail} samples')
print(f'Total time: {time.time()-t0:.1f}s')

### STEP 15: Save Batch Summary

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 4 — Step 15 — Save Summary')
print('='*60)

# Build aggregate dict keyed by factor name for JSON/CSV
agg_by_name = {}
for t_idx, name in enumerate(FACTOR_NAMES):
    if t_idx < len(agg_list):
        agg_by_name[name] = agg_list[t_idx]

summary = {
    'phase': 'Phase 4: SHAP',
    'experiment_id': EXP_ID,
    'split': SPLIT_NAME,
    'num_samples': n_avail,
    'n_background': N_BACKGROUND,
    'text_dim': TEXT_DIM,
    'fused_dim': FUSED_DIM,
    'method': 'DeepExplainer',
    'aggregate_modality': agg_by_name,
}
summary_path = os.path.join(XAI_OUT_DIR, 'shap_batch_summary.json')
save_raw_values(summary, summary_path)

# Save modality contribution CSV
if agg_by_name:
    rows = []
    for name in FACTOR_NAMES:
        if name in agg_by_name:
            a = agg_by_name[name]
            rows.append({
                'target': name,
                'mean_text_origin_pct': round(a.get('text_pct', 0), 1),
                'mean_image_origin_pct': round(a.get('image_pct', 0), 1),
            })
    if rows:
        csv_path = os.path.join(XAI_OUT_DIR, 'modality_contribution_summary.csv')
        pd.DataFrame(rows).to_csv(csv_path, index=False)
        print(f'Saved: {csv_path}')

print(f'Done ({time.time()-t0:.1f}s)')

### STEP 16: Reproducibility Check

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 4 — Step 16 — Reproducibility Check')
print('='*60)

wrapper = FusionHeadWrapper(model.head, score_index=0)
sv1, bv1 = compute_shap_values(wrapper, background.to(device), demo_fused.to(device))
sv2, bv2 = compute_shap_values(wrapper, background.to(device), demo_fused.to(device))

is_identical = np.allclose(sv1, sv2, atol=1e-6)
max_diff = np.abs(sv1 - sv2).max()
print(f'Max diff      : {max_diff:.2e}')
print(f'Base values   : {bv1:.4f} vs {bv2:.4f}')
print(f'Identical     : {is_identical}')
print(f'Reproducibility: {"PASSED" if is_identical else "FAILED"}')

print(f'Done ({time.time()-t0:.1f}s)')

### STEP 17: Final Summary

In [ ]:
print('='*60)
print('  PHASE 4 SHAP — FINAL SUMMARY')
print('='*60)

artifact_count = sum(len(f) for _, _, f in os.walk(XAI_OUT_DIR))

print(f'  Experiment      : {EXP_ID}')
print(f'  Split           : {SPLIT_NAME}')
print(f'  Samples         : {n_avail}')
print(f'  Background      : {N_BACKGROUND}')
print(f'  Total artifacts : {artifact_count}')
print(f'  Output dir      : {XAI_OUT_DIR}')
print()

checks = [
    ('Model loaded',           True),
    ('Embeddings extracted',   val_fused is not None and val_fused.shape[1] == FUSED_DIM),
    ('Prediction reproduced',  repro_err < 1e-4),
    ('SHAP computed',          len(demo_contributions) == NUM_TARGETS),
    ('Modality contribution',  all(name in demo_contributions for name in FACTOR_NAMES)),
    ('Ablation check',         ablation is not None),
    ('Batch processing',       batch_result is not None),
    ('Reproducibility',        is_identical),
]

all_passed = True
for desc, passed in checks:
    s = 'PASSED' if passed else 'FAILED'
    if not passed: all_passed = False
    print(f'  [{s:6s}] {desc}')

# Print modality profile from aggregate list
if agg_list:
    print(f'\n  Modality Profile (N={n_avail} samples):')
    for t_idx, name in enumerate(FACTOR_NAMES):
        if t_idx < len(agg_list):
            t = agg_list[t_idx].get('text_pct', 0)
            i = agg_list[t_idx].get('image_pct', 0)
            dom = 'text-dominant' if t > 60 else 'image-dominant' if i > 60 else 'balanced'
            print(f'    {name:<10s}: text={t:.0f}% image={i:.0f}% ({dom})')

print('='*60)
if all_passed:
    print('  All checks PASSED. Phase 4 complete.')
else:
    print('  Some checks FAILED.')
print('='*60)